# 2. Podstawy Supertonic 3

Cel: poznać minimalny przepływ **JSON → walidacja → Supertonic → zapis WAV**.

Notebook zapisuje zarówno wejściowy JSON, jak i wygenerowany plik audio w katalogu `outputs/`. Pierwsze utworzenie obiektu `TTS` pobiera model do lokalnego cache użytkownika.

In [ ]:
from pathlib import Path
import json
import sys

from IPython.display import Audio, display
from supertonic import TTS

def find_project_dir() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, current / "302-tts-supertonic", *current.parents]:
        if (candidate / "requirements.txt").is_file() and (candidate / "webgui").is_dir():
            return candidate
    raise FileNotFoundError("Nie znaleziono katalogu 302-tts-supertonic.")

PROJECT_DIR = find_project_dir()
VENV_DIR = (PROJECT_DIR / ".venv").resolve()
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if Path(sys.prefix).resolve() != VENV_DIR:
    raise RuntimeError("Przełącz kernel na: Supertonic Workshop (.venv).")

print("Kernel:", sys.executable)
print("Wyniki:", OUTPUT_DIR)

## Pierwsze wejście JSON

Zmień tekst, głos (`F1`–`F5`, `M1`–`M5`) lub język. Kod `na` oznacza tryb neutralny, gdy język nie jest znany.

In [ ]:
BASIC_REQUEST = {
    "text": "Cześć! To jest pierwsza lokalna synteza mowy z Supertonic 3.",
    "voice": "F2",
    "language": "pl",
}

print(json.dumps(BASIC_REQUEST, ensure_ascii=False, indent=2))

## Załadowanie modelu i funkcja pomocnicza

Model jest ładowany raz. Funkcja zapisuje dokładnie użyte wejście do `.json`, generuje dźwięk i zapisuje `.wav`.

In [ ]:
SUPPORTED_VOICES = {f"F{i}" for i in range(1, 6)} | {f"M{i}" for i in range(1, 6)}
tts = TTS(auto_download=True)

def synthesize_request(payload: dict, stem: str) -> Path:
    text = str(payload.get("text", "")).strip()
    voice = str(payload.get("voice", "F2")).upper()
    language = str(payload.get("language", "pl")).lower()

    if not text:
        raise ValueError("Pole text nie może być puste.")
    if voice not in SUPPORTED_VOICES:
        raise ValueError(f"Nieobsługiwany głos: {voice}")

    normalized = {"text": text, "voice": voice, "language": language}
    json_path = OUTPUT_DIR / f"{stem}.json"
    wav_path = OUTPUT_DIR / f"{stem}.wav"
    json_path.write_text(json.dumps(normalized, ensure_ascii=False, indent=2), encoding="utf-8")

    style = tts.get_voice_style(voice_name=voice)
    wav, duration = tts.synthesize(text=text, voice_style=style, lang=language)
    tts.save_audio(wav, str(wav_path))

    print(f"JSON: {json_path.name}")
    print(f"WAV:  {wav_path.name} | czas audio: {float(duration[0]):.2f} s")
    return wav_path

In [ ]:
basic_wav = synthesize_request(BASIC_REQUEST, "01-pierwsza-synteza")
display(Audio(filename=str(basic_wav)))

## Rozpoznawanie i test interpunkcji: `.`, `?`, `!`

Supertonic 3 obsługuje kropkę, znak zapytania i wykrzyknik. Znaki te są także granicami zdań podczas automatycznego dzielenia dłuższego tekstu. Intonacja jest jednak wynikiem działania modelu — sam znak nie gwarantuje zawsze tak samo mocnego efektu.

**Zadanie:** wygeneruj ten sam tekst najpierw z kropką, następnie z `?`, a na końcu z `!`. Posłuchaj, czy zmienia się melodia, akcent końcowy albo długość wypowiedzi.

In [ ]:
PUNCTUATION_TESTS = {
    "kropka": "Czy Supertonic odczyta to zdanie.",
    "pytanie": "Czy Supertonic odczyta to zdanie?",
    "wykrzyknienie": "Czy Supertonic odczyta to zdanie!",
}

for name, text in PUNCTUATION_TESTS.items():
    is_supported, unsupported = tts.model.text_processor.validate_text(text)
    print(f"{name:16} wspierane={is_supported} niewspierane={unsupported}")

In [ ]:
for name, text in PUNCTUATION_TESTS.items():
    request = {"text": text, "voice": "F2", "language": "pl"}
    punctuation_wav = synthesize_request(request, f"interpunkcja-{name}")
    print(name, "→", text)
    display(Audio(filename=str(punctuation_wav)))

### Dodatkowe próby

Powtórz porównanie dla przecinka, dwukropka, średnika, wielokropka i cudzysłowu. Następnie sprawdź liczby, skróty (np. `dr`, `itd.`), polskie znaki oraz emoji. Przed syntezą użyj `validate_text()`, ponieważ preprocessing może zamienić, pominąć albo odrzucić część znaków.

## Miejsce na własny eksperyment

Uzupełnij własny JSON. Wynik zostanie zapisany osobno, dzięki czemu można porównać wejścia i nagrania.

In [ ]:
OWN_REQUEST = {
    "text": "Tutaj wpisz własny tekst do przeczytania.",
    "voice": "M1",
    "language": "pl",
}

own_wav = synthesize_request(OWN_REQUEST, "02-wlasny-eksperyment")
display(Audio(filename=str(own_wav)))
# TWOJA IMPLEMENTACJA: zmień zawartość OWN_REQUEST i uruchom ponownie komórkę, aby wygenerować własny plik audio.

## Pytania warsztatowe

1. Jak zmienia się brzmienie po wyborze innego głosu?
2. Co dzieje się po podaniu niewłaściwego kodu języka?
3. Dlaczego zapisujemy także wejściowy JSON?
4. Która część kodu odpowiada za model, a która za zapis pliku?
5. Czy `?` i `!` zmieniły intonację w porównaniu z kropką?
6. Które dodatkowe znaki przeszły `validate_text()`, ale nie dały wyraźnej zmiany brzmienia?